# Reveal Telemetry Dataset EDA


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve()))
from src.ingestion.reveal import load_metadata, filter_telemetry_columns, load_run_telemetry


## 1. Workload and Phase Metadata


In [ ]:
df_cpu, df_gpu = load_metadata("../data/raw")
cpu_phases = df_cpu["phase"].value_counts()
gpu_phases = df_gpu["phase"].value_counts()
cpu_tasks = df_cpu["task"].value_counts()


#### Plot: Metadata Distributions


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cpu_phases.plot(kind="bar", ax=axes[0], color="#2b5c8f", title="CPU Phases")
gpu_phases.plot(kind="bar", ax=axes[1], color="#4682b4", title="GPU Phases")
cpu_tasks.plot(kind="barh", ax=axes[2], color="#3a7d44", title="CPU Workload Tasks")
for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


## 2. Sequence Length Distribution


In [ ]:
cpu_lengths = df_cpu["n_rows"]
gpu_lengths = df_gpu["n_rows"]
stats_df = pd.DataFrame({
    "CPU_Length": cpu_lengths.describe(),
    "GPU_Length": gpu_lengths.describe()
})


#### Plot: Sequence Lengths


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cpu_lengths, bins=40, color="#2b5c8f", edgecolor="black", alpha=0.8)
axes[0].set_title("CPU Run Lengths (Samples)")
axes[0].set_yscale("log")
axes[0].set_xlabel("Number of Telemetry Rows")
axes[0].set_ylabel("Log Count")

axes[1].hist(gpu_lengths, bins=40, color="#d95f02", edgecolor="black", alpha=0.8)
axes[1].set_title("GPU Run Lengths (Samples)")
axes[1].set_yscale("log")
axes[1].set_xlabel("Number of Telemetry Rows")
axes[1].set_ylabel("Log Count")

for ax in axes:
    ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 3. High-Frequency Telemetry Ingestion


In [ ]:
sample_path = "../data/raw/sample_run.csv.gz"
raw_cols = pd.read_csv(sample_path, compression="gzip", nrows=1).columns.tolist()
selected_cols = filter_telemetry_columns(raw_cols)
if "Unnamed: 0" in raw_cols and "Unnamed: 0" not in selected_cols:
    selected_cols.insert(0, "Unnamed: 0")

df_telemetry = load_run_telemetry(sample_path, selected_columns=selected_cols)
telemetry_summary = df_telemetry.describe().T[["mean", "std", "min", "max"]]


#### Plot: Hardware Telemetry Time Series


In [ ]:
time_idx = np.arange(len(df_telemetry))
plot_cols = [c for c in selected_cols if c != "Unnamed: 0"][:4]

fig, axes = plt.subplots(len(plot_cols), 1, figsize=(12, 2.5 * len(plot_cols)), sharex=True)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for idx, (col, color) in enumerate(zip(plot_cols, colors)):
    axes[idx].plot(time_idx, df_telemetry[col], color=color, linewidth=1.5)
    clean_title = col.split("[")[0].replace("htc-g003", "").strip(".")
    unit = col[col.find("["):col.find("]")+1] if "[" in col else ""
    axes[idx].set_ylabel(f"{clean_title} {unit}", fontsize=9)
    axes[idx].grid(True, linestyle="--", alpha=0.5)

axes[-1].set_xlabel("Time Step (Index)")
plt.tight_layout()
plt.show()


## 4. Subsystem Signal Correlations


In [ ]:
numeric_cols = [c for c in selected_cols if c != "Unnamed: 0"][:8]
corr_matrix = df_telemetry[numeric_cols].corr()
clean_labels = [c.split(":")[1].split(".")[0] for c in numeric_cols]


#### Plot: Telemetry Correlation Heatmap


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(clean_labels)))
ax.set_yticks(range(len(clean_labels)))
ax.set_xticklabels(clean_labels, rotation=45, ha="right")
ax.set_yticklabels(clean_labels)
plt.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Hardware Telemetry Cross-Correlation")
plt.tight_layout()
plt.show()
